# 🌿 AgriVision AI — التدريب السحابي على Google Colab (T4 GPU)
### تدريب نموذج تشخيص أمراض النباتات بالتعلم العميق وتصديره مباشرة بصيغة ONNX للموبايل

> **AgriVision AI**: منصة الذكاء الاصطناعي الزراعي الذكي لتشخيص أمراض المحاصيل (طماطم، بطاطس، ذرة، عنب، تفاح، وغيرها - 38 صنفاً مرضي وسليم).
> هذا النوت بوك مصمم ليعمل باستقلالية تامة وبضغطة واحدة على **Google Colab** باستخدام كرت شاشة مجاني **T4 GPU** وتنزيل النماذج المخرجة لجهازك تلقائياً.

## 1️⃣ التحقق من توفر كارت الشاشة (T4 GPU) وتثبيت الحزم

In [ ]:
!pip install -q timm albumentations onnx onnxruntime kaggle

import os, sys, time, json, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] Compute Device : {device}')
if torch.cuda.is_available():
    print(f'[*] GPU Name       : {torch.cuda.get_device_name(0)}')
    print(f'[*] GPU Memory     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('[!] تحذير: الـ GPU غير مفعل! يرجى الذهاب إلى Runtime -> Change runtime type واختيار T4 GPU.')

## 2️⃣ تنزيل مجموعة البيانات الزراعية مباشرة (PlantVillage - 38 صنفاً)

In [ ]:
# إعداد توكن كاجل لتحميل البيانات بسرعة فائقة (سرعة خوادم جوجل تفوق 100 ميجابايت/ثانية)
os.environ['KAGGLE_USERNAME'] = os.environ.get('KAGGLE_USERNAME', 'YOUR_KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = os.environ.get('KAGGLE_KEY', 'YOUR_KAGGLE_API_KEY')

dataset_dir = Path('/content/plant_dataset')
if not dataset_dir.exists():
    print('[*] جاري تحميل حزمة البيانات الزراعية من كاجل فك الضغط تلقائياً...')
    !kaggle datasets download -d vipoooool/new-plant-diseases-dataset --unzip -p /content/plant_dataset
    print('[+] تم تحميل وتجهيز البيانات بنجاح!')
else:
    print('[*] حزمة البيانات موجودة مسبقاً.')

# البحث الذكي عن مسارات التدريب والتحقق
train_dir = None
valid_dir = None

for root, dirs, files in os.walk(dataset_dir):
    lower_map = {d.lower(): d for d in dirs}
    if 'train' in lower_map and 'valid' in lower_map:
        train_dir = Path(root) / lower_map['train']
        valid_dir = Path(root) / lower_map['valid']
        break

print(f'[*] مسار التدريب : {train_dir}')
print(f'[*] مسار التحقق  : {valid_dir}')

# استخراج أسماء الأمراض والمحاصيل
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
num_classes = len(classes)
print(f'[*] عدد الأصناف والأمراض المكتشفة: {num_classes}')

class_to_idx = {cls: i for i, cls in enumerate(classes)}
idx_to_class = {i: cls for i, cls in enumerate(classes)}

with open('/content/class_indices.json', 'w', encoding='utf-8') as f:
    json.dump({'class_to_idx': class_to_idx, 'idx_to_class': idx_to_class}, f, indent=2)
print('[+] تم حفظ معجم الأصناف class_indices.json بنجاح.')

## 3️⃣ تجهيز محمل البيانات (DataLoaders) مع التحسينات الحقلية

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=str(train_dir), transform=train_transform)
valid_dataset = ImageFolder(root=str(valid_dir), transform=valid_transform)

BATCH_SIZE = 64 if torch.cuda.is_available() else 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'[*] عينات التدريب : {len(train_dataset)} صورة')
print(f'[*] عينات التحقق  : {len(valid_dataset)} صورة')
print(f'[*] حجم الدفعة   : {BATCH_SIZE}')

## 4️⃣ بناء النموذج (MobileNetV3) لنقل التعلم وخفة التنفيذ

In [ ]:
# تحميل نموذج MobileNetV3-Large ذو الكفاءة العالية للأجهزة المحمولة
model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

# تجميد الطبقات الأولى لتسريع التدريب
for param in model.features[:10].parameters():
    param.requires_grad = False

# رأس التصنيف المخصص لـ AgriVision AI
in_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.Hardswish(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)
)

model = model.to(device)
print(f'[+] تم تهيئة النموذج مع {num_classes} صنف بنجاح على {device}')

## 5️⃣ حلقة التدريب السريع بدقة Mixed Precision

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

EPOCHS = 10
best_val_acc = 0.0
best_model_path = '/content/agrivision_plant_vision_best.pth'
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('=' * 65)
print('      بدء تدريب نموذج AgriVision AI على كارت الشاشة T4 GPU      ')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        train_correct += preds.eq(labels).sum().item()
        train_total += labels.size(0)
        
        if (batch_idx + 1) % 200 == 0 or (batch_idx + 1) == len(train_loader):
            print(f'  [دورة {epoch:02d}/{EPOCHS:02d}] دفعة {batch_idx+1:04d}/{len(train_loader):04d} | دقة التدريب: {train_correct/train_total*100:.2f}% | Loss: {train_loss/train_total:.4f}')
            
    scheduler.step()
    
    # تقييم التحقق (Validation)
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            val_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)
            
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    elapsed = time.time() - start_time
    
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f'==> دورة [{epoch:02d}/{EPOCHS:02d}] في ({elapsed:.1f} ثانية) | دقة التحقق: {epoch_val_acc*100:.2f}% | خسارة التحقق: {epoch_val_loss:.4f}')
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'accuracy': best_val_acc,
            'classes': classes
        }, best_model_path)
        print(f'   --> [*] تم حفظ أفضل نموذج بدقة: {best_val_acc*100:.2f}%')

print(f'\n[+] اكتمل التدريب بنجاح! أعلى دقة تحقق مسجلة: {best_val_acc*100:.2f}%')

## 6️⃣ تصدير النموذج إلى ONNX المخصص للموبايل والتحقق منه

In [ ]:
!pip install -q onnxscript onnx

onnx_output_path = '/content/agrivision_plant_model.onnx'
ckpt = torch.load(best_model_path, map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
model_cpu = model.to('cpu')

dummy_input = torch.randn(1, 3, 224, 224, requires_grad=False)

print('[*] جاري تصدير النموذج بصيغة ONNX المجهزة لتطبيق Flutter الموبايل...')
try:
    torch.onnx.export(
        model_cpu,
        dummy_input,
        onnx_output_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input_image'],
        output_names=['class_probabilities'],
        dynamic_axes={
            'input_image': {0: 'batch_size'},
            'class_probabilities': {0: 'batch_size'}
        }
    )
except Exception as e:
    torch.onnx.export(
        model_cpu,
        dummy_input,
        onnx_output_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input_image'],
        output_names=['class_probabilities'],
        dynamic_axes={
            'input_image': {0: 'batch_size'},
            'class_probabilities': {0: 'batch_size'}
        },
        dynamo=False
    )

file_size_mb = os.path.getsize(onnx_output_path) / (1024 * 1024)
print(f'[+] تم تصدير النموذج بنجاح! الحجم: {file_size_mb:.2f} ميجابايت (خفيف جداً ومناسب للموبايل).')

# رسم وحفظ منحنيات التعلم
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', color='green')
plt.plot(history['val_loss'], label='Val Loss', color='red', linestyle='--')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot([a * 100 for a in history['train_acc']], label='Train Acc %', color='green')
plt.plot([a * 100 for a in history['val_acc']], label='Val Acc %', color='blue', linestyle='--')
plt.title('Accuracy Curve (%)')
plt.legend()
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=200)
plt.show()


## 7️⃣ تنزيل حزمة النماذج والمخرجات تلقائياً إلى جهازك 📥

In [ ]:
from google.colab import files
import zipfile

zip_filename = '/content/agrivision_ai_model_pack.zip'
artifacts = [
    '/content/agrivision_plant_model.onnx',
    '/content/class_indices.json',
    '/content/agrivision_plant_vision_best.pth',
    '/content/training_curves.png'
]

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for f in artifacts:
        if os.path.exists(f):
            zipf.write(f, arcname=os.path.basename(f))
            print(f'[+] تمت إضافة {os.path.basename(f)} إلى الحزمة.')

print('\n[*] جاري بدء تحميل الحزمة إلى حاسوبك تلقائياً عبر المتصفح...')
files.download(zip_filename)